# Silver Layer — ICU datetimeevents

## READINESS: partly profiled, rules drafted, awaiting remaining profiling and team approval

The type, missingness, range and relationship counts recorded in **Profiling Findings**
come from the run of 2026-10-19 (saved outputs of the previous version of this notebook).
Checks marked *not yet profiled* must be run and recorded before approval. Until the
`profiling_approved` widget is set to `true`, the publish step raises an error and nothing
is written. The dependency-ordered runner excludes this notebook by default.

**Known publish blocker.** The 2026-10-19 run passed every validation gate but
`createOrReplace` failed with `UC_DEPENDENCY_DOES_NOT_EXIST` ("Table
'cdr_mimiciv.silver.datetimeevents' is invalid because one of the underlying resources does
not exist"). The table currently in Silver dates from 2026-10-13 and has no
`_silver_processed_at`. The broken target must be investigated before publishing.

## Source
`<source_schema>.datetimeevents` (default `cdr_mimiciv.bronze.datetimeevents`)

## Target
`<target_schema>.datetimeevents` (default `cdr_mimiciv.silver.datetimeevents`), published
with CREATE OR REPLACE semantics. No tables are dropped.

## Grain (official MIMIC-IV icu documentation, `datetimeevents`)
One row per charted date observation for a patient during an ICU stay (for example the
date of last dialysis or a line insertion date). The documentation defines **no unique row
identifier**. Documented fields: `subject_id`, `hadm_id`, `stay_id` (patient, hospital stay
and ward stay identifiers; links to patients, admissions and icustays), `caregiver_id` (the
caregiver who documented the data), `charttime` (time the observation was charted; usually
the closest proxy to when it was measured), `storetime` (time the observation was manually
input or manually validated by clinical staff), `itemid` (measurement type; links to
`d_items`), `value` (the documented date itself, e.g. for itemid 225755 "18 Gauge
Insertion Date" the date the line was inserted), `valueuom` (documented as "almost always
the text string 'Date'") and `warning` (whether a warning was manually documented by the
care provider).
The documentation does not state NOT NULL constraints for any column.
[DOC] All dates are shifted for de-identification; each patient's chronology is preserved.

## Silver transformation (derived from the profiling findings)
Principle, as in admissions, labevents and chartevents: preserve source value, identify
anomaly, add a derived column or DQ flag, let downstream users decide.
- Select every Bronze column unchanged, in its Bronze type and order. (The previous version
  re-cast every column through `TRY_CAST(TRIM(CAST(... AS STRING)))` and narrowed `warning`
  to SMALLINT; Bronze is already typed and 0 cast failures were profiled, so the re-cast is
  removed and `warning` keeps its Bronze type.)
- Add DQ flag (our engineering choice), non-null boolean:
  - `dq_storetime_before_charttime`: `storetime < charttime`. Informational, not asserted
    to be an error; same name and meaning as labevents, chartevents and outputevents.
    Count *not yet profiled*.
- Add `_silver_processed_at`.
- No deduplication, no NULL filling, no timestamp correction, no row removal. `value` is
  never overwritten, and no interval (e.g. line days) is computed: that is a Gold feature.

## Open profiling questions
1. Are there exact duplicate rows? (No documented key.)
2. Values of `valueuom` and `warning`.
3. Timestamp order: `storetime` before `charttime`; `value` later than `charttime`
   (a documented date in the future of the charting time).
4. Relationships (observational only): `(stay_id, subject_id, hadm_id)` in icustays;
   `itemid` in `d_items` with `linksto = 'datetimeevents'`; `caregiver_id` in caregiver.

## Cost note
About 10 million rows. Profiling uses one aggregate pass, relationship checks broadcast the
small reference tables, no `exceptAll` is used, and the published table is checked with one
aggregate pass.

In [0]:
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")
dbutils.widgets.text("profiling_approved", "false")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"
PROFILING_APPROVED = dbutils.widgets.get("profiling_approved").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

from pyspark.sql import functions as F

SOURCE_TABLE = f"{BRONZE}.datetimeevents"
TARGET_TABLE = f"{SILVER}.datetimeevents"

DOCUMENTED_COLUMNS = [
    "subject_id", "hadm_id", "stay_id", "caregiver_id", "charttime", "storetime",
    "itemid", "value", "valueuom", "warning"
]

In [0]:
datetimeevents_bronze = spark.table(SOURCE_TABLE)
SOURCE_COLUMNS = datetimeevents_bronze.columns

datetimeevents_bronze.printSchema()
print("Documented columns missing from Bronze:", [c for c in DOCUMENTED_COLUMNS if c not in SOURCE_COLUMNS])
print("Bronze columns not in documentation:", [c for c in SOURCE_COLUMNS if c not in DOCUMENTED_COLUMNS])
print("Target exists:", spark.catalog.tableExists(TARGET_TABLE))
display(datetimeevents_bronze.limit(10))

root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- stay_id: integer (nullable = true)
 |-- caregiver_id: integer (nullable = true)
 |-- charttime: timestamp (nullable = true)
 |-- storetime: timestamp (nullable = true)
 |-- itemid: integer (nullable = true)
 |-- value: timestamp (nullable = true)
 |-- valueuom: string (nullable = true)
 |-- warning: integer (nullable = true)

Documented columns missing from Bronze: []
Bronze columns not in documentation: []
Target exists: True


subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valueuom,warning
10000032,29079034,39553978,18704,2180-07-23T14:24:00.000Z,2180-07-23T14:24:00.000Z,225754,2180-07-23T14:24:00.000Z,Date,0
10000032,29079034,39553978,18704,2180-07-23T14:24:00.000Z,2180-07-23T14:24:00.000Z,225755,2180-07-23T14:24:00.000Z,Date,0
10000032,29079034,39553978,20925,2180-07-23T21:02:00.000Z,2180-07-23T21:02:00.000Z,225754,2180-07-23T00:00:00.000Z,Date,0
10000032,29079034,39553978,20925,2180-07-23T21:02:00.000Z,2180-07-23T21:02:00.000Z,225755,2180-07-23T00:00:00.000Z,Date,0
10000690,25860671,37081114,17393,2150-11-02T20:00:00.000Z,2150-11-03T04:53:00.000Z,225754,2150-11-02T04:53:00.000Z,Date,0
10000690,25860671,37081114,17393,2150-11-02T20:38:00.000Z,2150-11-02T20:38:00.000Z,225755,2150-11-02T20:38:00.000Z,Date,0
10000690,25860671,37081114,17393,2150-11-03T01:01:00.000Z,2150-11-03T01:01:00.000Z,225755,2150-11-02T00:00:00.000Z,Date,0
10000690,25860671,37081114,17393,2150-11-03T04:52:00.000Z,2150-11-03T04:52:00.000Z,225755,2150-11-02T00:00:00.000Z,Date,0
10000690,25860671,37081114,33125,2150-11-03T08:14:00.000Z,2150-11-03T09:14:00.000Z,225754,2150-11-02T00:00:00.000Z,Date,0
10000690,25860671,37081114,33125,2150-11-03T08:14:00.000Z,2150-11-03T09:14:00.000Z,225755,2150-11-02T00:00:00.000Z,Date,0


In [0]:
# single-pass profile: missingness of every column, ranges and timestamp order
def _cnt(cond):
    return F.sum(F.when(cond, 1).otherwise(0))

profile_exprs = [F.count("*").alias("total_rows")]
profile_exprs += [_cnt(F.col(c).isNull()).alias(f"null_{c}") for c in SOURCE_COLUMNS]
profile_exprs += [
    _cnt(F.trim(F.col("valueuom")) == "").alias("valueuom_empty_string"),
    _cnt(F.col("storetime") < F.col("charttime")).alias("storetime_before_charttime"),
    _cnt(F.col("storetime") == F.col("charttime")).alias("storetime_equals_charttime"),
    _cnt(F.col("value") > F.col("charttime")).alias("value_after_charttime"),
]
for c in ("charttime", "storetime", "value"):
    profile_exprs += [F.min(c).alias(f"min_{c}"), F.max(c).alias(f"max_{c}")]

dt_profile = datetimeevents_bronze.agg(*profile_exprs).first().asDict()
for k, v in dt_profile.items():
    print(f"{k}: {v}")

total_rows: 9979761
null_subject_id: 0
null_hadm_id: 0
null_stay_id: 0
null_caregiver_id: 0
null_charttime: 0
null_storetime: 0
null_itemid: 0
null_value: 0
null_valueuom: 0
valueuom_empty_string: 0
storetime_before_charttime: 459636
storetime_equals_charttime: 4992530
value_after_charttime: 354932
min_charttime: 2110-01-11 12:40:00
max_charttime: 2214-07-26 08:00:00
min_storetime: 2110-01-11 12:41:00
max_storetime: 2214-07-26 09:20:00
min_value: 2109-08-02 00:00:00
max_value: 2214-07-24 09:57:00


In [0]:
# exact duplicate rows: no documented key, so the only grain check is on all columns
duplicate_groups = (
    datetimeevents_bronze.groupBy(*SOURCE_COLUMNS).count().filter(F.col("count") > 1)
)
dup_summary = duplicate_groups.agg(
    F.count("*").alias("duplicate_groups"),
    F.sum(F.col("count") - 1).alias("surplus_rows"),
).first().asDict()
print("Exact duplicates:", dup_summary)
if dup_summary["duplicate_groups"]:
    display(duplicate_groups.orderBy(F.desc("count")).limit(50))

Exact duplicates: {'duplicate_groups': 0, 'surplus_rows': None}


In [0]:
# categorical values (small result sets)
for c in ["valueuom", "warning"]:
    print(f"=== {c} ===")
    display(datetimeevents_bronze.groupBy(c).count().orderBy(F.desc("count")))

# examples of value later than charttime (inspect, do not remove)
display(
    datetimeevents_bronze
    .where(F.col("value") > F.col("charttime"))
    .groupBy("itemid").count()
    .orderBy(F.desc("count"))
    .limit(20)
)

=== valueuom ===


valueuom,count
Date,7153915
Date and Time,2825846


=== warning ===


warning,count
0,9976940
1,2821


itemid,count
225754,35612
224279,24292
224287,21686
225755,20636
224288,19891
224290,18289
224261,14387
224282,14116
224280,14001
224186,10682


In [0]:
# relationships (observational only; findings are recorded, not turned into flags here)
icustays_table = f"{SILVER}.icustays"
d_items_table = f"{SILVER}.d_items"
caregiver_table = f"{SILVER}.caregiver"

for _t in (icustays_table, d_items_table, caregiver_table):
    assert spark.catalog.tableExists(_t), f"Missing reference: {_t}"

icu_ref = (
    spark.table(icustays_table)
    .select("stay_id", "subject_id", "hadm_id", F.lit(True).alias("_stay_match"))
)
item_ref = spark.table(d_items_table).select("itemid", F.col("linksto").alias("_linksto"))
# caregiver_id is a string in Silver caregiver; cast to INT for the comparison
caregiver_ref = (
    spark.table(caregiver_table)
    .select(F.expr("TRY_CAST(trim(caregiver_id) AS INT)").alias("caregiver_id"))
    .where(F.col("caregiver_id").isNotNull())
    .distinct()
    .withColumn("_caregiver_match", F.lit(True))
)

rel = (
    datetimeevents_bronze.select("stay_id", "subject_id", "hadm_id", "caregiver_id", "itemid")
    .join(F.broadcast(icu_ref), ["stay_id", "subject_id", "hadm_id"], "left")
    .join(F.broadcast(item_ref), "itemid", "left")
    .join(F.broadcast(caregiver_ref), "caregiver_id", "left")
    .agg(
        _cnt(F.col("_stay_match").isNull()).alias("rows_stay_not_in_icustays"),
        _cnt(F.col("_linksto").isNull()).alias("rows_itemid_not_in_d_items"),
        _cnt(F.col("_linksto").isNotNull() & (F.col("_linksto") != "datetimeevents")).alias("rows_itemid_linksto_other_table"),
        _cnt(F.col("caregiver_id").isNotNull() & F.col("_caregiver_match").isNull()).alias("rows_caregiver_not_in_caregiver"),
    )
    .first().asDict()
)
for k, v in rel.items():
    print(f"{k}: {v}")

rows_stay_not_in_icustays: 0
rows_itemid_not_in_d_items: 0
rows_itemid_linksto_other_table: 0
rows_caregiver_not_in_caregiver: 0


## Profiling Findings

Recorded from the profiling run of 2026-10-19 on `cdr_mimiciv.bronze.datetimeevents`
(saved outputs of the previous notebook version). Rows marked *not yet profiled* must be
filled from the cells above before `profiling_approved` is set.
[DOC] = stated in official MIMIC documentation; [OURS] = our engineering choice.

| Check | Result | Silver treatment |
|---|---:|---|
| Rows | 9,979,761 | Row count preserved |
| Unique row key | None documented | No key asserted; no deduplication |
| Exact duplicate rows | *not yet profiled* | Report only; never removed [OURS] |
| Type casts of Bronze columns | 0 failures | Bronze types passed through; re-cast removed [OURS] |
| NULL in every column | 0 | None |
| `charttime` range | 2110-01-11 12:40 to 2214-07-26 08:00 | None; [DOC] dates shifted |
| `storetime` range | 2110-01-11 12:41 to 2214-07-26 09:20 | None |
| `value` range | 2109-08-02 to 2214-07-24 09:57 | None; a documented date may precede the ICU stay |
| `valueuom` values | *not yet profiled* | No change; [DOC] almost always "Date" |
| `warning` values | *not yet profiled* | No change |
| `storetime` before `charttime` | *not yet profiled* | `dq_storetime_before_charttime` [OURS]; informational, no correction |
| `value` after `charttime` | *not yet profiled* | Report only; decide after profiling |
| `stay_id` not in icustays / identity mismatch | 0 / 0 | None |
| `itemid` not in `d_items` | 0 | None |
| `itemid` with `linksto` other than datetimeevents | *not yet profiled* | Decide after profiling |
| `caregiver_id` not in caregiver | 0 | None |


In [0]:
# Silver transformation: all source columns unchanged + DQ flag
# (our engineering choice, see Profiling Findings); no rows removed, no source value changed
datetimeevents_silver = datetimeevents_bronze.select(
    *SOURCE_COLUMNS,
    F.coalesce(F.col("storetime") < F.col("charttime"), F.lit(False))
        .alias("dq_storetime_before_charttime"),
    F.current_timestamp().alias("_silver_processed_at"),
)

expected_schema = {f.name: f.dataType.simpleString() for f in datetimeevents_bronze.schema.fields}
expected_schema.update({
    "dq_storetime_before_charttime": "boolean",
    "_silver_processed_at": "timestamp",
})

# profiled expected counts (2026-10-19 run); add dq_storetime_before_charttime once profiled
PROFILED_COUNTS = {
    "silver_rows": 9979761,
}

datetimeevents_silver.printSchema()

root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- stay_id: integer (nullable = true)
 |-- caregiver_id: integer (nullable = true)
 |-- charttime: timestamp (nullable = true)
 |-- storetime: timestamp (nullable = true)
 |-- itemid: integer (nullable = true)
 |-- value: timestamp (nullable = true)
 |-- valueuom: string (nullable = true)
 |-- warning: integer (nullable = true)
 |-- dq_storetime_before_charttime: boolean (nullable = false)
 |-- _silver_processed_at: timestamp (nullable = false)



In [0]:
# publish gate: nothing below runs until profiling has been reviewed and approved
if not PROFILING_APPROVED:
    raise RuntimeError(
        "datetimeevents is not approved for publication: run and review the profiling cells, "
        "fill the 'not yet profiled' findings and any justified rules, resolve the broken "
        "target table, then set widget profiling_approved=true."
    )

# cheap pre-publish validation (no data scan)
pre_schema = {f.name: f.dataType.simpleString() for f in datetimeevents_silver.schema.fields}
assert list(pre_schema.items()) == list(expected_schema.items()), "Unexpected Silver schema"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

(
    datetimeevents_silver.writeTo(TARGET_TABLE)
    .using("delta")
    .createOrReplace()
)

print(f"Published: {TARGET_TABLE}")

Published: cdr_mimiciv.silver.datetimeevents


In [0]:
# post-publish validation: one aggregate pass over Silver, one count of Bronze (no exceptAll)
published = spark.table(TARGET_TABLE)

post_exprs = [
    F.count("*").alias("silver_rows"),
    _cnt(F.col("_silver_processed_at").isNull()).alias("null_processed_at"),
    _cnt(F.col("dq_storetime_before_charttime").isNull()).alias("null_dq_flags"),
    _cnt(F.col("dq_storetime_before_charttime")).alias("dq_storetime_before_charttime"),
    _cnt(
        F.col("dq_storetime_before_charttime")
        != F.coalesce(F.col("storetime") < F.col("charttime"), F.lit(False))
    ).alias("flag_mismatches"),
]
post_exprs += [_cnt(F.col(c).isNull()).alias(f"null_{c}") for c in SOURCE_COLUMNS]
post = published.agg(*post_exprs).first().asDict()
bronze_rows = datetimeevents_bronze.count()

published_schema = {f.name: f.dataType.simpleString() for f in published.schema.fields}

print("Bronze rows:", bronze_rows)
print("Published:", post)

assert post["silver_rows"] == bronze_rows, "Silver row count differs from Bronze"
assert post["null_processed_at"] == 0
assert post["null_dq_flags"] == 0, "DQ flags must be non-null booleans"
assert post["flag_mismatches"] == 0, "DQ flags do not match their definitions"
assert list(published_schema.items()) == list(expected_schema.items()), \
    f"Schema mismatch: {published_schema}"
# published NULL counts equal the Bronze profile (source values passed through)
for c in SOURCE_COLUMNS:
    assert post[f"null_{c}"] == dt_profile[f"null_{c}"], f"NULL count changed for {c}"

if ENFORCE_PROFILED_COUNTS:
    for name, expected in PROFILED_COUNTS.items():
        assert post[name] == expected, f"{name}: expected {expected}, got {post[name]}"

dbutils.notebook.exit("datetimeevents:OK")